# Notebook 04-2 — Dense + BM25 2-way Grid Search (실무 시나리오)

**목적**: BGE-M3 Dense + 전통 BM25 조합의 최적 가중치 탐색  
**배경**: RAG 실무에서 native sparse/ColBERT 없이 Dense + BM25만으로 hybrid 구성하는 관행을 반영  

**코드 4 대비 차이점**:
- 3-way (Dense + native Sparse + ColBERT) 대신 2-way (Dense + BM25)
- 코드 4에서 구축된 BM25 캐시와 Dense 캐시를 재활용
- Score cache를 새로 구축 (BM25 raw score + Dense score on candidates)
- 231개 가중치 exhaustive grid (0.05 단위)

**Inputs**:
- 코드 4의 BM25 캐시: `2nd_exp/cache/notebook04_3way_fullcorpus/{year}/bm25_pool/`
- 코드 4의 Dense 캐시: `2nd_exp/cache/notebook04_3way_fullcorpus/{year}/bge_dense/`
- `2nd_exp/data/{year}/queries_final.csv`

**Outputs**:
- `2nd_exp/results/notebook04_2way_grid/grid_cellwise_*.csv`
- `2nd_exp/results/notebook04_2way_grid/best_weights_*.csv`
- `2nd_exp/results/notebook04_2way_grid/grid_summary_*.csv`
- `2nd_exp/results/notebook04_2way_grid/details/*.csv`

In [2]:
import os, re, json, time, pickle, random, hashlib, gc
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional, Any

import numpy as np
import pandas as pd
import torch
import faiss

from rank_bm25 import BM25Okapi
from FlagEmbedding import BGEM3FlagModel

# ----------------------------
# 0) Global settings
# ----------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)

# ----------------------------
# 1) Settings
# ----------------------------
BASE_DIR = "2nd_exp"
DATA_DIR = os.path.join(BASE_DIR, "data")

OUT_DIR = os.path.join(BASE_DIR, "results", "notebook04_2way_grid")
os.makedirs(OUT_DIR, exist_ok=True)

# 코드 4의 기존 캐시 재활용
NB04_CACHE_DIR = os.path.join(BASE_DIR, "cache", "notebook04_3way_fullcorpus")

# 2-way 전용 score cache
NB04_2WAY_CACHE_DIR = os.path.join(BASE_DIR, "cache", "notebook04_2way_fullcorpus")
os.makedirs(NB04_2WAY_CACHE_DIR, exist_ok=True)

YEARS = [2025]
QUERY_TYPES = ["summary", "keyword", "function"]

TEXT_FIELD = "abstract"
DENSE_MAX_LEN = 512

POOL_TOP_P_BM25 = 200
POOL_TOP_P_DENSE = 200

EVAL_TOP_K = 50

METRICS = ["Hit@10", "MRR@10", "Hit@50"]

SCORE_SAVE_EVERY_N_QUERIES = 50

DENSE_DOC_BATCH = 32
DENSE_QUERY_BATCH = 1

# Raw full-corpus CSV
RAW_INPUT_FILES = {
    2005: "independent_claims_y2005_with_abstract_wipo_cpc.csv",
    2015: "independent_claims_y2015_with_abstract_wipo_cpc.csv",
    2025: "independent_claims_y2025_with_abstract_wipo_cpc.csv",
}

# ----------------------------
# 2) 2-way Weight grid: 0.05 단위, w_dense + w_bm25 = 1.0
# ----------------------------
STEP = 0.05
WEIGHT_GRID_2WAY = []
for wd_int in range(0, 21):
    wb_int = 20 - wd_int
    WEIGHT_GRID_2WAY.append((
        round(wd_int * STEP, 2),
        round(wb_int * STEP, 2),
    ))
assert len(WEIGHT_GRID_2WAY) == 21
print(f"2-way weight grid: {len(WEIGHT_GRID_2WAY)} combinations")
print(f"  range: Dense [{WEIGHT_GRID_2WAY[0][0]}..{WEIGHT_GRID_2WAY[-1][0]}], BM25 [{WEIGHT_GRID_2WAY[0][1]}..{WEIGHT_GRID_2WAY[-1][1]}]")

# ----------------------------
# 3) Utility functions
# ----------------------------
def normalize_text(s):
    if s is None: return ""
    s = str(s).replace("\r", " ").replace("\n", " ")
    return re.sub(r"\s+", " ", s).strip()

def find_first_existing_col(df, candidates):
    cols_lower = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in cols_lower:
            return cols_lower[cand.lower()]
    return None

def simple_tokenize(text):
    return re.findall(r"[a-z0-9]+", str(text).lower())

def ensure_dir(p):
    os.makedirs(p, exist_ok=True)

def atomic_write_json(obj, path):
    tmp = path + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)
    for attempt in range(5):
        try:
            os.replace(tmp, path)
            return
        except PermissionError:
            time.sleep(0.5)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)
    if os.path.exists(tmp):
        try: os.remove(tmp)
        except Exception: pass

def atomic_write_pickle(obj, path):
    tmp = path + ".tmp"
    with open(tmp, "wb") as f:
        pickle.dump(obj, f)
    for attempt in range(5):
        try:
            os.replace(tmp, path)
            return
        except PermissionError:
            time.sleep(0.5)
    with open(path, "wb") as f:
        pickle.dump(obj, f)
    if os.path.exists(tmp):
        try: os.remove(tmp)
        except Exception: pass

def read_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def read_pickle(path):
    with open(path, "rb") as f:
        return pickle.load(f)

def sha1_of_strings(items, max_items=200, stride=5):
    h = hashlib.sha1()
    n = len(items)
    h.update(str(n).encode("utf-8"))
    if n == 0: return h.hexdigest()
    idxs = list(range(0, n, stride))
    if len(idxs) > max_items: idxs = idxs[:max_items]
    for i in idxs:
        s = str(items[i]) if items[i] is not None else ""
        h.update(str(i).encode("utf-8"))
        h.update(str(len(s)).encode("utf-8"))
        h.update(s[:200].encode("utf-8", errors="ignore"))
        h.update(s[-200:].encode("utf-8", errors="ignore"))
    return h.hexdigest()

def sha1_of_queries(df_q):
    h = hashlib.sha1()
    h.update(str(len(df_q)).encode("utf-8"))
    for _, r in df_q.head(500).iterrows():
        h.update(str(r["patent_id"]).encode("utf-8"))
        h.update(str(r["query_type"]).encode("utf-8"))
        qtx = str(r["query_text"])
        h.update(str(len(qtx)).encode("utf-8"))
        h.update(qtx[:200].encode("utf-8", errors="ignore"))
    return h.hexdigest()

# ----------------------------
# 4) Metrics
# ----------------------------
def hit_at_k(ranked_ids, gold_id, k):
    return 1.0 if gold_id in ranked_ids[:k] else 0.0

def mrr_at_k(ranked_ids, gold_id, k):
    for i, pid in enumerate(ranked_ids[:k], start=1):
        if pid == gold_id:
            return 1.0 / i
    return 0.0

def metrics_for_query(ranked_ids, gold_id):
    return {
        "Hit@10": hit_at_k(ranked_ids, gold_id, 10),
        "MRR@10": mrr_at_k(ranked_ids, gold_id, 10),
        "Hit@50": hit_at_k(ranked_ids, gold_id, 50),
    }

# ----------------------------
# 5) Full corpus data loading
# ----------------------------
@dataclass
class YearData:
    year: int
    patent_ids: List[str]
    docs_field: List[str]
    n_docs: int
    pid_to_idx: Dict[str, int]
    docs_hash: str
    df_queries: pd.DataFrame
    queries_hash: str

def load_full_corpus_texts(year):
    fname = RAW_INPUT_FILES[year]
    path = os.path.join(os.getcwd(), fname)
    if not os.path.exists(path):
        raise FileNotFoundError(f"[{year}] Raw corpus not found: {fname}")
    print(f"[{year}] Loading FULL corpus: {fname}")
    df = pd.read_csv(path, dtype=str, keep_default_na=False, na_values=[], low_memory=False)

    claims_col = find_first_existing_col(df, ["claims", "independent_claims", "independent_claim", "claim"])
    abstract_col = find_first_existing_col(df, ["abstract", "abstract_text", "abs"])
    id_col = find_first_existing_col(df, ["patent_id", "publication_number", "pub_number", "doc_number", "patent_number", "id"])

    if claims_col != "claims": df = df.rename(columns={claims_col: "claims"})
    if abstract_col != "abstract": df = df.rename(columns={abstract_col: "abstract"})
    if id_col != "patent_id": df = df.rename(columns={id_col: "patent_id"})

    df["patent_id"] = df["patent_id"].astype(str).str.strip().str.replace(r"\.0$", "", regex=True)
    df["claims"] = df["claims"].fillna("").map(normalize_text)
    df["abstract"] = df["abstract"].fillna("").map(normalize_text)

    n_before = len(df)
    df = df.drop_duplicates(subset="patent_id", keep="first").reset_index(drop=True)
    if len(df) < n_before:
        print(f"[{year}] Removed {n_before - len(df)} duplicate patent_ids")

    patent_ids = df["patent_id"].tolist()
    docs = df[TEXT_FIELD].tolist()
    print(f"[{year}] Full corpus ready: n_docs={len(patent_ids):,}")
    return patent_ids, docs

def load_queries(year):
    candidates = [
        os.path.join(DATA_DIR, str(year), "queries_final.csv"),
        os.path.join(DATA_DIR, str(year), "queries.csv"),
    ]
    qpath = None
    for p in candidates:
        if os.path.exists(p): qpath = p; break
    if qpath is None:
        raise FileNotFoundError(f"[{year}] queries not found")
    dfq = pd.read_csv(qpath, dtype={"patent_id": str})
    dfq["patent_id"] = dfq["patent_id"].astype(str).str.strip().str.replace(r"\.0$", "", regex=True)
    dfq["query_type"] = dfq["query_type"].astype(str).str.lower().str.strip()
    dfq["query_text"] = dfq["query_text"].fillna("").astype(str)
    dfq = dfq[dfq["query_type"].isin(QUERY_TYPES)].reset_index(drop=True)
    return dfq

def load_year_data(year):
    patent_ids, docs = load_full_corpus_texts(year)
    dfq = load_queries(year)
    pid_to_idx = {pid: i for i, pid in enumerate(patent_ids)}
    mask = dfq["patent_id"].isin(pid_to_idx)
    n_bad = (~mask).sum()
    if n_bad > 0:
        print(f"[{year}] WARNING: {n_bad} queries removed (gold not in corpus)")
    dfq = dfq[mask].reset_index(drop=True)
    docs_hash = sha1_of_strings(docs)
    queries_hash = sha1_of_queries(dfq)
    return YearData(year=year, patent_ids=patent_ids, docs_field=docs,
                    n_docs=len(patent_ids), pid_to_idx=pid_to_idx,
                    docs_hash=docs_hash, df_queries=dfq, queries_hash=queries_hash)

# ----------------------------
# 6) BM25: 코드 4 캐시 재활용 / Dense: 코드 3 캐시 재활용
# ----------------------------
NB03_CACHE_DIR = os.path.join(BASE_DIR, "cache", "notebook03_rankcache_fullcorpus")

def load_existing_bm25(year):
    """Load BM25 tokenized corpus from NB04 cache."""
    pkl_path = os.path.join(NB04_CACHE_DIR, str(year), "bm25_pool", f"bm25_{TEXT_FIELD}.pkl")
    if not os.path.exists(pkl_path):
        raise FileNotFoundError(f"[BM25][{year}] cache not found: {pkl_path}")
    obj = read_pickle(pkl_path)
    bm25 = BM25Okapi(obj["tok"])
    print(f"[BM25][{year}] loaded from NB04 cache")
    return bm25

def safe_name(x):
    return re.sub(r"[^a-zA-Z0-9_\\-]+", "_", str(x))

def load_existing_dense_nb03(year):
    """Load Dense FAISS index + embeddings from NB03 cache."""
    root = os.path.join(NB03_CACHE_DIR, str(year), "dense_index")
    enc_safe = safe_name("BAAI/bge-m3")
    index_path = os.path.join(root, f"faiss__{enc_safe}__{TEXT_FIELD}__ML{DENSE_MAX_LEN}.index")
    emb_path = os.path.join(root, f"docemb__{enc_safe}__{TEXT_FIELD}__ML{DENSE_MAX_LEN}.mmap")
    if not os.path.exists(index_path):
        raise FileNotFoundError(f"[DENSE][{year}] NB03 index not found: {index_path}")
    if not os.path.exists(emb_path):
        raise FileNotFoundError(f"[DENSE][{year}] NB03 emb not found: {emb_path}")
    idx = faiss.read_index(index_path)
    n = idx.ntotal
    dim = idx.d
    emb = np.memmap(emb_path, dtype="float32", mode="r", shape=(n, dim))
    print(f"[DENSE][{year}] loaded from NB03 cache (n={n:,}, dim={dim})")
    return idx, emb

# ----------------------------
# 7) 2-way Score cache: Dense + BM25
# ----------------------------
def score_cache_2way_paths(year, qtype):
    root = os.path.join(NB04_2WAY_CACHE_DIR, "grid_scores", str(year), qtype)
    ensure_dir(root)
    tag = f"{TEXT_FIELD}_denseL{DENSE_MAX_LEN}_P{POOL_TOP_P_BM25}_D{POOL_TOP_P_DENSE}"
    return {
        "root": root,
        "meta": os.path.join(root, f"meta_{tag}.json"),
        "progress": os.path.join(root, f"progress_{tag}.json"),
        "data": os.path.join(root, f"scores_{tag}.pkl"),
    }

def build_2way_score_cache(yd, bm25_obj, dense_index, dense_emb, dense_model, qtype):
    paths = score_cache_2way_paths(yd.year, qtype)
    exp = {
        "year": yd.year, "text_field": TEXT_FIELD, "dense_max_len": DENSE_MAX_LEN,
        "pool_bm25": POOL_TOP_P_BM25, "pool_dense": POOL_TOP_P_DENSE,
        "docs_hash": yd.docs_hash, "queries_hash": yd.queries_hash,
        "mode": "2way_dense_bm25",
    }

    # Check existing cache
    if os.path.exists(paths["data"]) and os.path.exists(paths["meta"]):
        try:
            meta = read_json(paths["meta"])
            if all(meta.get(k) == exp.get(k) for k in ["year", "text_field", "docs_hash", "queries_hash", "mode"]):
                obj = read_pickle(paths["data"])
                print(f"[2WAY][{yd.year}][{qtype}] cache OK (n={len(obj['qids'])}) -> skip")
                return obj
        except Exception:
            pass

    df_sub = yd.df_queries[yd.df_queries["query_type"] == qtype].reset_index(drop=True)
    n_q = len(df_sub)
    if n_q == 0:
        obj = {"qids": [], "gold_pid": [], "cand_idx": [], "dense_scores": [], "bm25_scores": []}
        atomic_write_pickle(obj, paths["data"])
        atomic_write_json(exp, paths["meta"])
        return obj

    # Resume
    start_q = 0
    obj = {"qids": [], "gold_pid": [], "cand_idx": [], "dense_scores": [], "bm25_scores": []}
    if os.path.exists(paths["progress"]) and os.path.exists(paths["data"]):
        try:
            prog = read_json(paths["progress"])
            start_q = int(prog.get("next_q_index", 0))
            obj = read_pickle(paths["data"])
            start_q = min(start_q, len(obj.get("qids", [])))
            print(f"[2WAY][{yd.year}][{qtype}] resume: start_q={start_q}/{n_q}")
        except Exception:
            start_q = 0
            obj = {"qids": [], "gold_pid": [], "cand_idx": [], "dense_scores": [], "bm25_scores": []}

    def bm25_topk(query, k):
        scores = bm25_obj.get_scores(simple_tokenize(query))
        return np.argsort(scores)[::-1][:k].tolist()

    def bm25_score_cands(query, cand_idx):
        all_scores = bm25_obj.get_scores(simple_tokenize(query))
        return np.array([all_scores[i] for i in cand_idx], dtype=np.float32)

    def dense_topk(query, k):
        qv = dense_model.encode([query], batch_size=1, max_length=DENSE_MAX_LEN)
        if isinstance(qv, dict): qv = qv["dense_vecs"]
        qv = np.asarray(qv, dtype=np.float32)
        faiss.normalize_L2(qv)
        _, idx = dense_index.search(qv, k)
        return idx[0].tolist()

    def dense_score_cands(query, cand_idx):
        qv = dense_model.encode([query], batch_size=1, max_length=DENSE_MAX_LEN)
        if isinstance(qv, dict): qv = qv["dense_vecs"]
        qv = np.asarray(qv, dtype=np.float32).flatten()
        faiss.normalize_L2(qv.reshape(1, -1))
        doc = np.asarray(dense_emb[cand_idx], dtype=np.float32)
        return (doc @ qv).astype(np.float32)

    print(f"[2WAY][{yd.year}][{qtype}] building score cache (n_queries={n_q})...")
    t0 = time.time()

    for qi in range(start_q, n_q):
        row = df_sub.iloc[qi]
        gold = str(row["patent_id"])
        qtext = str(row["query_text"])

        # Candidate pool: BM25 top-P ∪ Dense top-P
        cand = set(bm25_topk(qtext, POOL_TOP_P_BM25))
        if POOL_TOP_P_DENSE > 0:
            cand.update(dense_topk(qtext, POOL_TOP_P_DENSE))
        cand_idx = sorted(list(cand))

        # Dense scores on candidates
        d_scores = dense_score_cands(qtext, cand_idx)

        # BM25 scores on candidates
        b_scores = bm25_score_cands(qtext, cand_idx)

        obj["qids"].append(int(qi))
        obj["gold_pid"].append(gold)
        obj["cand_idx"].append(cand_idx)
        obj["dense_scores"].append(d_scores)
        obj["bm25_scores"].append(b_scores)

        if ((qi + 1) % SCORE_SAVE_EVERY_N_QUERIES == 0) or (qi + 1 == n_q):
            atomic_write_pickle(obj, paths["data"])
            atomic_write_json({"next_q_index": qi + 1, "n_queries": n_q}, paths["progress"])
            elapsed = (time.time() - t0) / 60
            print(f"[2WAY][{yd.year}][{qtype}] {qi+1}/{n_q} ({elapsed:.1f} min)")

    atomic_write_pickle(obj, paths["data"])
    atomic_write_json(exp, paths["meta"])
    if os.path.exists(paths["progress"]):
        os.remove(paths["progress"])
    print(f"[2WAY][{yd.year}][{qtype}] done.")
    return obj

# ----------------------------
# 8) 2-way fusion: min-max norm + weighted sum
# ----------------------------
def minmax_norm(arr):
    mn, mx = arr.min(), arr.max()
    if mx - mn < 1e-12:
        return np.zeros_like(arr)
    return (arr - mn) / (mx - mn)

def rank_from_scores(cand_idx, score):
    ord_ = np.argsort(score)[::-1]
    return [cand_idx[i] for i in ord_[:EVAL_TOP_K]]

def evaluate_2way_weights(patent_ids, cache_obj, w):
    wd, wb = w
    n = len(cache_obj["qids"])
    if n == 0:
        return {m: np.nan for m in METRICS}
    vals = {m: [] for m in METRICS}
    for i in range(n):
        d_norm = minmax_norm(cache_obj["dense_scores"][i])
        b_norm = minmax_norm(cache_obj["bm25_scores"][i])
        fused = wd * d_norm + wb * b_norm
        ranked_doc_idx = rank_from_scores(cache_obj["cand_idx"][i], fused)
        ranked_pids = [patent_ids[j] for j in ranked_doc_idx]
        gold = cache_obj["gold_pid"][i]
        ms = metrics_for_query(ranked_pids, gold)
        for m in METRICS:
            vals[m].append(ms[m])
    return {m: float(np.mean(vals[m])) for m in METRICS}

# ----------------------------
# 9) Weight selection (Option A)
# ----------------------------
def summarize_2way_weight(df_cell, w):
    sub = df_cell[
        (df_cell["weight_dense"] == w[0]) &
        (df_cell["weight_bm25"] == w[1])
    ].copy()
    overall = {m: float(sub[m].mean()) for m in METRICS}
    worst_year = {}
    for m in METRICS:
        per_year = sub.groupby("year")[m].mean()
        worst_year[m] = float(per_year.min())
    year_macro = sub.groupby("year")[METRICS].mean().reset_index()
    return {
        "w": w,
        **{f"Overall_{m}": overall[m] for m in METRICS},
        **{f"WorstYear_{m}": worst_year[m] for m in METRICS},
        "year_macro": year_macro
    }

def tie_break_key(s):
    return (s["Overall_Hit@10"], s["WorstYear_Hit@10"], s["Overall_MRR@10"],
            s["WorstYear_MRR@10"], s["Overall_Hit@50"], s["WorstYear_Hit@50"])

def to_row_dict_2way(tag, s):
    w = s["w"]
    row = {"tag": tag, "weight_dense": w[0], "weight_bm25": w[1]}
    for m in METRICS:
        row[f"Overall_{m}"] = s[f"Overall_{m}"]
        row[f"WorstYear_{m}"] = s[f"WorstYear_{m}"]
    return row

# ============================================================
# 10) MAIN EXECUTION
# ============================================================
print("\n=== PHASE 0: Load Year Data ===")
year_data = {y: load_year_data(y) for y in YEARS}
for y in YEARS:
    yd = year_data[y]
    print(f"[{y}] docs={yd.n_docs:,} | queries={len(yd.df_queries)}")

print("\n=== PHASE 1: Load BM25 (from NB04) + Dense (from NB03) ===")
bm25_objs = {}
dense_indices = {}
dense_embs = {}

for y in YEARS:
    bm25_objs[y] = load_existing_bm25(y)
    dense_indices[y], dense_embs[y] = load_existing_dense_nb03(y)

# Load BGE-M3 model for query encoding only
print("\nLoading BGE-M3 model for query encoding...")
dense_model = BGEM3FlagModel("BAAI/bge-m3", use_fp16=True, device=DEVICE)

print("\n=== PHASE 2: Build 2-way Score Caches ===")
score_caches = {}
for y in YEARS:
    for qt in QUERY_TYPES:
        obj = build_2way_score_cache(
            yd=year_data[y],
            bm25_obj=bm25_objs[y],
            dense_index=dense_indices[y],
            dense_emb=dense_embs[y],
            dense_model=dense_model,
            qtype=qt
        )
        score_caches[(y, qt)] = obj

# Recall@pool
print("\n=== Recall@pool ===")
for y in YEARS:
    yd = year_data[y]
    for qt in QUERY_TYPES:
        obj = score_caches[(y, qt)]
        n = len(obj["qids"])
        hits = sum(1 for i in range(n)
                   if yd.pid_to_idx.get(obj["gold_pid"][i]) in obj["cand_idx"][i])
        print(f"  [{y}][{qt}] Recall@pool = {hits/n:.4f} ({hits}/{n})")

# Free model memory
dense_model = None
torch.cuda.empty_cache(); gc.collect()

print("\n=== PHASE 3: Evaluate 2-way Weight Grid ===")
rows = []
t0 = time.time()
for w in WEIGHT_GRID_2WAY:
    for y in YEARS:
        yd = year_data[y]
        for qt in QUERY_TYPES:
            met = evaluate_2way_weights(yd.patent_ids, score_caches[(y, qt)], w)
            rows.append({
                "weight_dense": w[0], "weight_bm25": w[1],
                "year": y, "query_type": qt, **met
            })
    elapsed = (time.time() - t0) / 60
    print(f"[2WAY-GRID] done weight={w} | {elapsed:.1f} min")

df_cell = pd.DataFrame(rows)
tag = f"{TEXT_FIELD}_denseL{DENSE_MAX_LEN}_P{POOL_TOP_P_BM25}_D{POOL_TOP_P_DENSE}"
cell_out = os.path.join(OUT_DIR, f"grid_cellwise_2way_{tag}.csv")
df_cell.to_csv(cell_out, index=False)
print("Saved:", cell_out)

# Best global
summaries = [summarize_2way_weight(df_cell, w) for w in WEIGHT_GRID_2WAY]
best_global = sorted(summaries, key=tie_break_key, reverse=True)[0]
print("\n=== BEST GLOBAL 2-WAY WEIGHT ===")
print(f"w(dense, bm25) = {best_global['w']}")
print({k: best_global[k] for k in best_global if k.startswith("Overall_") or k.startswith("WorstYear_")})

# Best per type
best_by_type = {}
for qt in QUERY_TYPES:
    sub_qt = df_cell[df_cell["query_type"] == qt]
    summaries_qt = [summarize_2way_weight(sub_qt, w) for w in WEIGHT_GRID_2WAY]
    best_by_type[qt] = sorted(summaries_qt, key=tie_break_key, reverse=True)[0]
    print(f"\n=== BEST 2-WAY for type={qt} ===")
    print(f"w = {best_by_type[qt]['w']}")

# Save outputs
summary_rows = [to_row_dict_2way("GLOBAL", best_global)] + \
               [to_row_dict_2way(f"TYPE_{qt}", best_by_type[qt]) for qt in QUERY_TYPES]
df_best = pd.DataFrame(summary_rows)
best_out = os.path.join(OUT_DIR, f"best_weights_2way_{tag}.csv")
df_best.to_csv(best_out, index=False)
print("\nSaved:", best_out)

detail_dir = os.path.join(OUT_DIR, "details")
os.makedirs(detail_dir, exist_ok=True)
best_global["year_macro"].to_csv(os.path.join(detail_dir, f"global_best_year_macro_2way_{TEXT_FIELD}.csv"), index=False)
for qt in QUERY_TYPES:
    best_by_type[qt]["year_macro"].to_csv(os.path.join(detail_dir, f"type_{qt}_best_year_macro_2way_{TEXT_FIELD}.csv"), index=False)
print("Saved detail tables under:", detail_dir)

all_rows = [to_row_dict_2way("GRID", s) for s in summaries]
df_grid_summary = pd.DataFrame(all_rows).sort_values("Overall_Hit@10", ascending=False)
grid_sum_out = os.path.join(OUT_DIR, f"grid_summary_2way_{tag}.csv")
df_grid_summary.to_csv(grid_sum_out, index=False)
print("Saved:", grid_sum_out)

print("\nNotebook 04-2 completed.")
print("Outputs:", OUT_DIR)

DEVICE: cuda
2-way weight grid: 21 combinations
  range: Dense [0.0..1.0], BM25 [1.0..0.0]

=== PHASE 0: Load Year Data ===
[2025] Loading FULL corpus: independent_claims_y2025_with_abstract_wipo_cpc.csv
[2025] Full corpus ready: n_docs=240,989
[2025] docs=240,989 | queries=3150

=== PHASE 1: Load BM25 (from NB04) + Dense (from NB03) ===
[BM25][2025] loaded from NB04 cache
[DENSE][2025] loaded from NB03 cache (n=240,989, dim=1024)

Loading BGE-M3 model for query encoding...


Fetching 30 files: 100%|██████████| 30/30 [00:00<00:00, 7426.18it/s]



=== PHASE 2: Build 2-way Score Caches ===
[2WAY][2025][summary] building score cache (n_queries=1050)...


pre tokenize: 100%|██████████| 1/1 [00:00<00:00, 500.04it/s]
You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
Inference Embeddings: 100%|██████████| 1/1 [00:00<00:00, 13.07it/s]


[2WAY][2025][summary] 50/1050 (2.8 min)


...

[2WAY][2025][function] 950/1050 (54.8 min)


Inference Embeddings: 100%|██████████| 1/1 [00:00<00:00, 10.69it/s]


[2WAY][2025][function] 1000/1050 (57.8 min)


Inference Embeddings: 100%|██████████| 1/1 [00:00<00:00, 11.23it/s]


[2WAY][2025][function] 1050/1050 (60.8 min)
[2WAY][2025][function] done.

=== Recall@pool ===
  [2025][summary] Recall@pool = 0.9705 (1019/1050)
  [2025][keyword] Recall@pool = 0.9857 (1035/1050)
  [2025][function] Recall@pool = 0.9752 (1024/1050)

=== PHASE 3: Evaluate 2-way Weight Grid ===
[2WAY-GRID] done weight=(0.0, 1.0) | 0.0 min
[2WAY-GRID] done weight=(0.05, 0.95) | 0.0 min
[2WAY-GRID] done weight=(0.1, 0.9) | 0.0 min
[2WAY-GRID] done weight=(0.15, 0.85) | 0.0 min
[2WAY-GRID] done weight=(0.2, 0.8) | 0.0 min
[2WAY-GRID] done weight=(0.25, 0.75) | 0.0 min
[2WAY-GRID] done weight=(0.3, 0.7) | 0.0 min
[2WAY-GRID] done weight=(0.35, 0.65) | 0.0 min
[2WAY-GRID] done weight=(0.4, 0.6) | 0.0 min
[2WAY-GRID] done weight=(0.45, 0.55) | 0.0 min
[2WAY-GRID] done weight=(0.5, 0.5) | 0.0 min
[2WAY-GRID] done weight=(0.55, 0.45) | 0.0 min
[2WAY-GRID] done weight=(0.6, 0.4) | 0.0 min
[2WAY-GRID] done weight=(0.65, 0.35) | 0.0 min
[2WAY-GRID] done weight=(0.7, 0.3) | 0.0 min
[2WAY-GRID] done w